In [1]:
from decouple import AutoConfig
config = AutoConfig(search_path='./../.env')

In [2]:
import os
os.environ["COHERE_API_KEY"] = config('COHERE_TOKEN')

### Loading Embedding Model

In [ ]:
!ollama pull nomic-embed-text

In [4]:
from langchain_ollama import OllamaEmbeddings
model_name='nomic-embed-text'
embeddings = OllamaEmbeddings(model=model_name,)
embeddings

### Data Loading and Ingestion

In [6]:
from langchain_community.document_loaders import PyPDFLoader

file_dir = './../../../../data/nifty10_reports/'
file_name = 'TCS.pdf'

# Load, chunk and index the contents of the file.
loader = PyPDFLoader(os.path.join(file_dir, file_name))
pages = loader.load_and_split()

In [ ]:
pages = pages[10:20]
pages

#### Embedding and Storing documents into vectorstore

**Chunking/Splitting** the data into relevant chunks is very important as it helps decide what context will be passed for answer generation. It is advisable to carefully go through the data and identify the `seperators` and `chunk_size`.

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

separators=[" \n\n ", " \n "]
text_splitter = RecursiveCharacterTextSplitter(
    separators=separators,
    chunk_size=5000,
    chunk_overlap=100,
    length_function=len,
)

In [ ]:
docs = text_splitter.split_documents(pages)
len(docs)

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain.docstore.document import Document

if not os.path.exists('./doc'):
    os.mkdir('./doc')
Chroma().delete_collection()
vectorstore = Chroma.from_documents(docs, embeddings, persist_directory='./doc/chroma_db')

### Loaidng the VectorDB and Retriever

In [11]:
from langchain.vectorstores.base import VectorStoreRetriever
from langchain_community.vectorstores import Chroma

In [12]:

top_k_docs = 5
vectorstore_path = './doc/chroma_db'
vectorstore = Chroma(persist_directory=vectorstore_path, embedding_function=embeddings)
retriever = VectorStoreRetriever(vectorstore=vectorstore, search_kwargs={"k": top_k_docs})

##### With Cohere Reranker

In [13]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank

In [ ]:
compressor = CohereRerank()
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever,
    top_n=5
)

#### Semantic Search

In [ ]:
question = "Who is the CEO of TCS?"
top_k = vectorstore.similarity_search(question)
top_k

In [ ]:
retriever.get_relevant_documents(question)

## Generation Models

In [ ]:
!ollama pull "llama3.2"

In [ ]:
from langchain_ollama.llms import OllamaLLM

llama3 = OllamaLLM(model="llama3.2",
                         temperature=0.0,
                         max_tokens=4096,
                         top_k=10,)

llama3

### Answer Generation

In [23]:
from langchain.chains import RetrievalQAWithSourcesChain
from langchain.prompts import PromptTemplate

template = """You are an AI assistant for answering questions about the provided text.
If you don't know the answer, just say that you don't know, don't try to make up an answer. 
Use three sentences maximum and keep the answer as concise as possible. 
Always say "thanks for asking!" at the end of the answer. 
{summaries}
Question: {question}
Helpful Answer:"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

qa_src_chain = RetrievalQAWithSourcesChain.from_chain_type(
    llama3,
    retriever=retriever, return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT},
)

In [ ]:
question = "Who are the members in top managemnet of TCS?"
result = qa_src_chain.invoke({"question": question})
result

In [ ]:
print(result['answer'])

In [ ]:
question = "What are key highlights from the letter to shareholders?"
result = qa_src_chain.invoke({"question": question})
result

In [ ]:
question = "What are the key highlights from consolidated financial statement?"
result = qa_src_chain.invoke({"question": question})
result

P.S.: Running this notebook will help you realise the importance of efficient data parsing.